# 23 FINAL — Run & Blind-Score Frozen External Holdout v4.3

Notebook ini menjalankan **baseline WLC** dan **main-ingredient-aware WLC** pada 98 query yang sudah dibekukan.

Penting:
- Candidate retrieval dibuat **identik** untuk kedua metode.
- Baseline: `0.7 × semantic_minmax + 0.3 × harmonic_efficiency`.
- Proposed: `0.7 × semantic_minmax + 0.3 × (0.5 × harmonic_efficiency + 0.5 × main_ingredient_coverage)`.
- Top-5 kedua sistem dipool, dideduplikasi, lalu diacak secara blind untuk penilaian manual.
- Jangan membuka file mapping sistem/rank sampai semua grade selesai.

Evaluasi baru ini mengukur **kualitas/akseptabilitas output Top-5**, bukan keberhasilan menemukan satu recipe ID target tertentu dari full corpus.

In [1]:
# Jalankan di environment Dapuria yang memiliki chroma_db, model SBERT, dan dapuria_pipeline.py.
# Jika package belum ada:
# !pip -q install sentence-transformers chromadb scipy pandas matplotlib

import json, math, os, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd

from dapuria_pipeline import (
    DapuriaRecipeService,
    parse_user_ingredients,
    coverage_stats,
    harmonic_efficiency,
    MAIN_INGREDIENT_KEYWORDS,
    ingredient_matches,
)

QUERY_FILE = Path("external_eval_queries_v4_3.csv")
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "resep_masakan"
SEED = 20260928
TOP_N = 5

assert QUERY_FILE.exists(), (
    "external_eval_queries_v4_3.csv belum tersedia. Jalankan notebook 22 dulu."
)
queries = pd.read_csv(QUERY_FILE)
assert len(queries) == 98
print("Frozen queries:", len(queries))

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Frozen queries: 98


In [2]:
# Verify the exact frozen v4.3 holdout before any Dapuria retrieval is run.
MANIFEST_FILE = Path("external_eval_manifest_v4_3_FROZEN.json")
assert MANIFEST_FILE.exists(), (
    "external_eval_manifest_v4_3_FROZEN.json tidak ditemukan."
)

manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
actual_sha256 = hashlib.sha256(QUERY_FILE.read_bytes()).hexdigest()

print("Manifest version :", manifest.get("version"))
print("Manifest SHA-256:", manifest["csv_sha256"])
print("Actual SHA-256:  ", actual_sha256)

assert manifest.get("version") == "v4.3", "Manifest bukan versi v4.3."
assert manifest.get("manual_audit_status") == "PASS", (
    "Holdout belum lolos manual audit."
)
assert str(manifest.get("status", "")).startswith("frozen_"), (
    "Manifest belum berstatus frozen."
)
assert actual_sha256 == manifest["csv_sha256"], (
    "HASH MISMATCH: external_eval_queries_v4_3.csv berbeda dari frozen file."
)
assert manifest.get("n_queries") == 98
assert manifest.get("automated_quality_flags", 999) == 0
assert len(queries) == 98
assert queries["query_id"].nunique() == 98
assert queries["query_text"].nunique() == 98

# Reject duplicate ingredient combinations even if order differs.
query_sets = queries["query_items_json"].map(lambda z: tuple(sorted(json.loads(z))))
assert query_sets.nunique() == 98, "Duplicate unordered query set ditemukan."

print("Frozen v4.3 holdout verified. Aman lanjut retrieval.")

Manifest version : v4.3
Manifest SHA-256: 2ce61d31e8ed38937fdcd8528ee6cf197a13beadf5dde4e2b97186cee478ed61
Actual SHA-256:   2ce61d31e8ed38937fdcd8528ee6cf197a13beadf5dde4e2b97186cee478ed61
Frozen v4.3 holdout verified. Aman lanjut retrieval.


In [3]:
# Helpers for proposed main-ingredient coverage.
def is_main_ingredient_local(name):
    for kws in MAIN_INGREDIENT_KEYWORDS.values():
        if any(ingredient_matches(name, kw) for kw in kws):
            return True
    return False


def main_coverage(user_ingredients, recipe_ingredients, fallback_user_coverage):
    mains = [u for u in user_ingredients if is_main_ingredient_local(u)]
    if not mains:
        return float(fallback_user_coverage)
    matched = sum(
        1 for u in mains if any(ingredient_matches(u, r) for r in recipe_ingredients)
    )
    return matched / len(mains)


service = DapuriaRecipeService(chroma_dir=CHROMA_DIR, collection_name=COLLECTION_NAME)
print(service.healthcheck())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4562.07it/s]


{'model_name': 'LazarusNLP/all-indo-e5-small-v4', 'collection_name': 'resep_masakan', 'document_count': 5000, 'config': {'top_k_sbert': 50, 'max_keyword_extra': 20, 'w1': 0.7, 'w2': 0.3}}


In [4]:
# Compute both scores from exactly the same candidate pool.
def rank_both(query_text, top_n=5):
    user_ingredients = parse_user_ingredients(query_text)
    candidates = service.retrieve_candidates(query_text, user_ingredients)
    if not candidates:
        return [], []

    sem = np.asarray([c["sbert_score"] for c in candidates], dtype=float)
    smin, smax = float(sem.min()), float(sem.max())
    snorm = (sem - smin) / (smax - smin + 1e-9)

    rows = []
    for c, s in zip(candidates, snorm):
        st = coverage_stats(user_ingredients, c["ingredient_names"])
        eh = harmonic_efficiency(st["coverage_recipe"], st["coverage_user"])
        cm = main_coverage(user_ingredients, c["ingredient_names"], st["coverage_user"])
        em = 0.5 * eh + 0.5 * cm
        rows.append(
            {
                **c,
                **st,
                "semantic_norm": float(s),
                "eff_harmonic": float(eh),
                "main_coverage": float(cm),
                "eff_mainaware": float(em),
                "score_baseline": 0.7 * float(s) + 0.3 * eh,
                "score_proposed": 0.7 * float(s) + 0.3 * em,
            }
        )

    base = sorted(
        rows, key=lambda x: (-x["score_baseline"], -x["semantic_norm"], str(x["id"]))
    )[:top_n]
    prop = sorted(
        rows, key=lambda x: (-x["score_proposed"], -x["semantic_norm"], str(x["id"]))
    )[:top_n]
    return base, prop

In [5]:
# Run all 98 frozen queries and save explicit rankings.
ranking_rows = []
meta_by_qid = {}

for i, q in queries.iterrows():
    qid = q["query_id"]
    base, prop = rank_both(q["query_text"], TOP_N)
    if len(base) != TOP_N or len(prop) != TOP_N:
        print("WARNING", qid, "returned", len(base), len(prop), "results")
    meta_by_qid[qid] = {"baseline": base, "proposed": prop}
    for system, arr, score_key in [
        ("baseline", base, "score_baseline"),
        ("proposed", prop, "score_proposed"),
    ]:
        for rank, r in enumerate(arr, 1):
            ranking_rows.append(
                {
                    "query_id": qid,
                    "query_text": q["query_text"],
                    "target_main_category": q["target_main_category"],
                    "system": system,
                    "rank": rank,
                    "recipe_id": r["id"],
                    "recipe_title": r["title"],
                    "recipe_ingredients": json.dumps(
                        r["ingredient_names"], ensure_ascii=False
                    ),
                    "semantic_norm": r["semantic_norm"],
                    "eff_harmonic": r["eff_harmonic"],
                    "main_coverage": r["main_coverage"],
                    "final_score": r[score_key],
                }
            )

rankings = pd.DataFrame(ranking_rows)
rankings.to_csv("external_eval_rankings_v4_3.csv", index=False)
print("Ranking rows:", len(rankings))
rankings.head()

Ranking rows: 980


,query_id,query_text,target_main_category,system,rank,recipe_id,recipe_title,recipe_ingredients,semantic_norm,eff_harmonic,main_coverage,final_score
0,EXT43_001,"beras, ayam",Serealia,baseline,1,resep_4073,Nasi Ayam Thailand,"[""ayam"", ""beras"", ""daun ketumbar beserta akarn...",1.000000,0.461538,1.0,0.838462
1,EXT43_001,"beras, ayam",Serealia,baseline,2,resep_103,"Nasi Lembek, Nugget Ayam Tahu dan Timun (9 mon...","[""beras"", ""daging ayam giling"", ""timun"", ""tahu...",0.859222,0.444444,1.0,0.734789
2,EXT43_001,"beras, ayam",Serealia,baseline,3,resep_3310,Arem-Arem Ayam Sayur,"[""beras"", ""santan encer"", ""daun salam"", ""serai...",0.934748,0.222222,1.0,0.720990
3,EXT43_001,"beras, ayam",Serealia,baseline,4,resep_843,Nasi Soto Ayam,"[""g nasi putih"", ""g dada ayam rebus"", ""daun ba...",0.887987,0.285714,0.5,0.707305
4,EXT43_001,"beras, ayam",Serealia,baseline,5,resep_2276,Bubur Ayam Simpel,"[""g beras"", ""saset santan rose brand"", ""g ayam...",0.699067,0.666667,1.0,0.689347


In [6]:
# Pool Top-5 from BOTH systems, deduplicate by recipe ID, then blind-shuffle within query.
blind_rows = []
map_rows = []

for _, q in queries.iterrows():
    qid = q["query_id"]
    sub = rankings[rankings["query_id"] == qid]
    ids = sorted(
        sub["recipe_id"].unique(),
        key=lambda rid: hashlib.sha256(f"{SEED}|{qid}|{rid}".encode()).hexdigest(),
    )
    for j, rid in enumerate(ids, 1):
        rr = sub[sub["recipe_id"] == rid]
        exemplar = rr.iloc[0]
        blind_id = f"{qid}_C{j:02d}"
        blind_rows.append(
            {
                "query_id": qid,
                "query_text": q["query_text"],
                "blind_candidate_id": blind_id,
                "candidate_title": exemplar["recipe_title"],
                "candidate_ingredients": exemplar["recipe_ingredients"],
                "grade_0_1_2": "",
                "judgment_note": "",
            }
        )
        base_rank = rr.loc[rr["system"] == "baseline", "rank"]
        prop_rank = rr.loc[rr["system"] == "proposed", "rank"]
        map_rows.append(
            {
                "query_id": qid,
                "blind_candidate_id": blind_id,
                "recipe_id": rid,
                "baseline_rank": int(base_rank.iloc[0]) if len(base_rank) else "",
                "proposed_rank": int(prop_rank.iloc[0]) if len(prop_rank) else "",
            }
        )

blind = pd.DataFrame(blind_rows)
secret_map = pd.DataFrame(map_rows)
blind.to_csv("external_eval_pooled_blind_v4_3.csv", index=False)
secret_map.to_csv("external_eval_pooled_SECRET_MAP_v4_3.csv", index=False)
print("Blind judgments to label:", len(blind))
print(
    "\nJANGAN buka external_eval_pooled_SECRET_MAP_v4_3.csv sebelum labeling selesai."
)
print("Untuk labeling, gunakan HANYA external_eval_pooled_blind_v4_3.csv.")
print(
    "System identity, rank, score, dan target category tidak ditampilkan pada file blind."
)
blind.head(12)

Blind judgments to label: 585

JANGAN buka external_eval_pooled_SECRET_MAP_v4_3.csv sebelum labeling selesai.
Untuk labeling, gunakan HANYA external_eval_pooled_blind_v4_3.csv.
System identity, rank, score, dan target category tidak ditampilkan pada file blind.


,query_id,query_text,blind_candidate_id,candidate_title,candidate_ingredients,grade_0_1_2,judgment_note
0,EXT43_001,"beras, ayam",EXT43_001_C01,"Nasi Lembek, Nugget Ayam Tahu dan Timun (9 mon...","[""beras"", ""daging ayam giling"", ""timun"", ""tahu...",,
1,EXT43_001,"beras, ayam",EXT43_001_C02,Bubur Ayam Simpel,"[""g beras"", ""saset santan rose brand"", ""g ayam...",,
2,EXT43_001,"beras, ayam",EXT43_001_C03,Nasi Soto Ayam,"[""g nasi putih"", ""g dada ayam rebus"", ""daun ba...",,
3,EXT43_001,"beras, ayam",EXT43_001_C04,Nasi Ayam Thailand,"[""ayam"", ""beras"", ""daun ketumbar beserta akarn...",,
4,EXT43_001,"beras, ayam",EXT43_001_C05,Bubur Ayam,"[""beras"", ""garam"", ""santan instan"", ""air"", ""ay...",,
5,EXT43_001,"beras, ayam",EXT43_001_C06,Arem-Arem Ayam Sayur,"[""beras"", ""santan encer"", ""daun salam"", ""serai...",,
6,EXT43_002,"tepung maizena, wortel",EXT43_002_C01,Tahu isi,"[""tepung terigu"", ""tepung beras"", ""kubis"", ""wo...",,
7,EXT43_002,"tepung maizena, wortel",EXT43_002_C02,Dadar Jagung Tahu Tepung Maizena,"[""jagung manis"", ""tahu putih"", ""daun bawang"", ...",,
8,EXT43_002,"tepung maizena, wortel",EXT43_002_C03,Perkedel Mie Tahu,"[""mie instan"", ""tahu putih"", ""tepung serbaguna...",,
9,EXT43_002,"tepung maizena, wortel",EXT43_002_C04,Brokoli telur maizena,"[""brokoli yg ukurang sedang"", ""bawang putih"", ...",,


In [7]:
# Pre-labeling integrity checks.
assert len(rankings) == 98 * 5 * 2, f"Expected 980 ranking rows, got {len(rankings)}"
counts = rankings.groupby(["query_id", "system"]).size()
assert counts.eq(5).all(), "Ada query/system yang tidak memiliki tepat 5 hasil."

assert blind["grade_0_1_2"].eq("").all()
assert blind["judgment_note"].eq("").all()
assert blind[["query_id", "blind_candidate_id"]].duplicated().sum() == 0

pool_sizes = blind.groupby("query_id").size()
print("Ranking rows:", len(rankings))
print("Blind pooled candidates:", len(blind))
print(
    "Pool size per query: min =",
    int(pool_sizes.min()),
    "max =",
    int(pool_sizes.max()),
    "mean =",
    round(float(pool_sizes.mean()), 2),
)
print("Pre-labeling integrity checks: PASS")

Ranking rows: 980
Blind pooled candidates: 585
Pool size per query: min = 5 max = 9 mean = 5.97
Pre-labeling integrity checks: PASS


## Rubrik blind judgment

Isi kolom `grade_0_1_2` pada `external_eval_pooled_blind_v4_3.csv` **tanpa membuka** file SECRET MAP, rankings, score, atau output sistem lainnya:

- **0 — Not relevant / unsuitable:** kandidat tidak masuk akal sebagai rekomendasi untuk kombinasi bahan pada query.
- **1 — Acceptable / plausible:** kandidat masih masuk akal dan menggunakan kombinasi bahan query secara wajar, walaupun bukan kecocokan yang sangat kuat.
- **2 — Strongly relevant:** kandidat sangat sesuai dengan kombinasi bahan query; bahan utama dan konteks hidangan cocok dengan jelas.

Nilai **kecocokan resep yang ditampilkan**, bukan apakah kandidat sama dengan `source_title` yang dipakai untuk membentuk query.

Setelah selesai, simpan sebagai `external_eval_pooled_blind_v4_3_LABELED.csv`, baru lanjut ke sel scoring di bawah.

**Aturan blind:** selama labeling, lihat hanya `query_text`, `candidate_title`, dan `candidate_ingredients`.
Jangan memakai `source_title` dari dataset publik sebagai jawaban acuan; source recipe hanya dipakai untuk membentuk query.


In [8]:
# SCORING — jalankan hanya setelah blind labeling selesai.
LABELED = Path("external_eval_pooled_blind_v4_3_LABELED.csv")
MAP = Path("external_eval_pooled_SECRET_MAP_v4_3.csv")

if not LABELED.exists():
    print(
        "Belum ada file labeled. Selesaikan blind judgment lalu simpan sebagai:",
        LABELED,
    )
else:
    lab = pd.read_csv(LABELED)
    mp = pd.read_csv(MAP)
    lab["grade_0_1_2"] = pd.to_numeric(lab["grade_0_1_2"], errors="raise").astype(int)
    assert lab["grade_0_1_2"].isin([0, 1, 2]).all()
    judged = lab.merge(
        mp, on=["query_id", "blind_candidate_id"], how="inner", validate="one_to_one"
    )
    print("Judged candidates:", len(judged))

Belum ada file labeled. Selesaikan blind judgment lalu simpan sebagai: external_eval_pooled_blind_v4_3_LABELED.csv


In [9]:
# Metric functions and final summary.
def dcg(grades):
    return sum((2**g - 1) / math.log2(i + 2) for i, g in enumerate(grades))


def ndcg_within_output(grades):
    ideal = sorted(grades, reverse=True)
    denom = dcg(ideal)
    return dcg(grades) / denom if denom > 0 else 0.0


def wilson_interval(successes, n, z=1.959963984540054):
    if n == 0:
        return (float("nan"), float("nan"))
    p = successes / n
    den = 1 + z * z / n
    center = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return center - half, center + half


if LABELED.exists():
    records = []
    per_query = []
    for system, rank_col in [
        ("baseline", "baseline_rank"),
        ("proposed", "proposed_rank"),
    ]:
        for qid, g in judged[judged[rank_col].notna()].groupby("query_id"):
            g = g.copy()
            g[rank_col] = pd.to_numeric(g[rank_col]).astype(int)
            g = g.sort_values(rank_col)
            grades = g["grade_0_1_2"].tolist()
            assert len(grades) == 5, (system, qid, len(grades))
            rec = {
                "query_id": qid,
                "system": system,
                "acceptable_success_5": int(any(x >= 1 for x in grades)),
                "strong_success_5": int(any(x == 2 for x in grades)),
                "acceptable_precision_5": sum(x >= 1 for x in grades) / 5,
                "strong_precision_5": sum(x == 2 for x in grades) / 5,
                "within_output_ndcg_5": ndcg_within_output(grades),
            }
            per_query.append(rec)

    pq = pd.DataFrame(per_query)
    for system, g in pq.groupby("system"):
        n = len(g)
        acc_hits = int(g["acceptable_success_5"].sum())
        strong_hits = int(g["strong_success_5"].sum())
        lo, hi = wilson_interval(acc_hits, n)
        records.append(
            {
                "system": system,
                "n_queries": n,
                "acceptable_success_at_5": acc_hits / n,
                "acceptable_success_at_5_ci95_low": lo,
                "acceptable_success_at_5_ci95_high": hi,
                "strong_success_at_5": strong_hits / n,
                "acceptable_precision_at_5": g["acceptable_precision_5"].mean(),
                "strong_precision_at_5": g["strong_precision_5"].mean(),
                "within_output_ndcg_at_5": g["within_output_ndcg_5"].mean(),
            }
        )
    summary = pd.DataFrame(records)
    summary.to_csv("external_eval_results_summary_v4_3.csv", index=False)
    pq.to_csv("external_eval_results_per_query_v4_3.csv", index=False)
    display(summary)

In [10]:
# Paired exact McNemar for Top-5 success, plus result graph.
if LABELED.exists():
    from scipy.stats import binomtest
    import matplotlib.pyplot as plt

    wide = pq.pivot(index="query_id", columns="system", values="acceptable_success_5")
    b_only = int(((wide["baseline"] == 1) & (wide["proposed"] == 0)).sum())
    p_only = int(((wide["baseline"] == 0) & (wide["proposed"] == 1)).sum())
    disc = b_only + p_only
    p_mcnemar = (
        1.0
        if disc == 0
        else binomtest(
            min(b_only, p_only), n=disc, p=0.5, alternative="two-sided"
        ).pvalue
    )
    print("Acceptable Success@5 discordant pairs:")
    print(
        "baseline-only =",
        b_only,
        " proposed-only =",
        p_only,
        " exact McNemar p =",
        p_mcnemar,
    )

    metrics = [
        "acceptable_success_at_5",
        "strong_success_at_5",
        "acceptable_precision_at_5",
        "within_output_ndcg_at_5",
    ]
    labels = [
        "Acceptable\nSuccess@5",
        "Strong\nSuccess@5",
        "Acceptable\nP@5",
        "Within-output\nnDCG@5",
    ]
    x = np.arange(len(metrics))
    width = 0.36
    b = summary[summary.system == "baseline"].iloc[0]
    p = summary[summary.system == "proposed"].iloc[0]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.bar(x - width / 2, [b[m] for m in metrics], width, label="Baseline WLC")
    ax.bar(
        x + width / 2, [p[m] for m in metrics], width, label="Main-ingredient-aware WLC"
    )
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_xticks(x, labels)
    ax.legend()
    ax.set_title("External Blind Top-5 Output-Quality Evaluation")
    fig.tight_layout()
    fig.savefig("external_eval_results_bar_v4_3.png", dpi=300, bbox_inches="tight")
    plt.show()

## Interpretasi yang boleh dilaporkan

Gunakan nama metrik yang tepat. Misalnya, jika `acceptable_success_at_5 = 0.76`, laporkan sebagai **76% Acceptable Top-5 Success Rate** (atau istilah final yang disepakati dengan pembimbing), yaitu proporsi query yang memiliki sedikitnya satu rekomendasi Top-5 dengan grade ≥ 1.

**Jangan menyebut metrik ini sebagai strict ground-truth Hit@5** terhadap recipe ID tertentu. Ini adalah evaluasi blind terhadap kualitas output yang ditampilkan.

Jika pembimbing mensyaratkan “>70%”, pastikan dulu beliau menjelaskan **70% untuk metrik apa**. Jangan mengganti atau membuang query setelah melihat hasil untuk mengejar ambang tersebut.